# 01 · One shot

The whole incident goes in, one diagnosis comes out. This is the baseline every
later chapter is measured against — and the chapter where it fails.

> **You'll learn**
> - Call a reasoner on the control plane and read its structured answer
> - See a one-shot diagnosis land an incident correctly
> - Watch the same call fail on an incident that punishes the obvious story

In [1]:
import os, sys, json, time, pathlib, httpx
sys.path.insert(0, str(pathlib.Path.cwd().parent / "lib"))
import dag

SERVER = os.environ.get("AGENTFIELD_SERVER", "http://localhost:8080")
GT = {g["id"]: g for g in json.load(open("../incidents/ground_truth.json"))["incidents"]}

def run(reasoner, **inp):
    """Dispatch to the node over HTTP and wait. Returns (run_id, output).

    Async on purpose: it hands back the run_id the DAG needs, and it does not
    die at the control plane's 90s synchronous ceiling.
    (Never `await app.call(...)` from a notebook — it executes children twice.)
    """
    r = httpx.post(f"{SERVER}/api/v1/execute/async/blast-radius.{reasoner}",
                   json={"input": inp}, timeout=30).json()
    eid, rid = r["execution_id"], r["run_id"]
    for _ in range(200):
        time.sleep(3)
        d = httpx.get(f"{SERVER}/api/v1/executions/{eid}", timeout=20).json()
        d = d.get("data", d)
        if d.get("status") in ("succeeded", "completed"):
            return rid, d.get("output") or d.get("result")
        if d.get("status") in ("failed", "error"):
            raise RuntimeError(d.get("error"))
    raise TimeoutError(eid)

def show(dx, title=""):
    if title:
        print(title); print("=" * len(title))
    print("root cause  :", dx["root_cause"])
    print("remediation :", dx["remediation"])
    print("confident   :", dx["confident"])
    for f in dx["findings"]:
        print(f"  - [{f['severity']:<8}] {f['location']}: {f['claim']}")


`r01` is nine lines: read the incident, answer once, return a `Diagnosis`.
No loop, no second opinion, no tools.

In [2]:
print(open("../node/rungs/r01.py").read())

"""r01 — one shot.

The whole incident, one call, one schema. The baseline everything else is measured
against.
"""
from agentfield import AgentRouter

from common import MODEL, SYSTEM, Diagnosis, incident_text

router = AgentRouter(prefix="r01", tags=["rung", "r01"])


@router.reasoner(tags=["entry"])
async def diagnose(incident_id: str, model: str | None = None) -> Diagnosis:
    """Read everything, answer once."""
    return await router.app.ai(
        system=SYSTEM,
        user=incident_text(incident_id),
        schema=Diagnosis,
        model=model or MODEL,
    )



## 1 · An incident it solves

`inc-002` is a payments 5xx spike. The evidence points one way and the answer
is in the artifacts.

In [3]:
rid_002, d002 = run("r01_diagnose", incident_id="inc-002")
show(d002, "r01 on inc-002")

r01 on inc-002
root cause  : The payments-gateway 2.14.0 deploy introduced a bug in MoneyRounding.toMinorUnits that incorrectly assumes all currencies have scale=2, causing ArithmeticException for JPY (scale=0) on authorization requests.
remediation : Immediately roll back payments-gateway to version 2.13.x, or apply a hotfix to MoneyRounding.toMinorUnits to use the actual scale from CurrencyRegistry (scale=0 for JPY) instead of hardcoding scale=2.
confident   : True
  - [critical] logs: JPY authorization requests fail with ArithmeticException: Rounding necessary in MoneyRounding.toMinorUnits, with a warning about the scale assumption.
  - [high    ] metrics: JPY error rate spiked to 99.7% while EUR and overall other currencies remained near 0.1%, isolating the impact to JPY transactions.
  - [high    ] deploys: The multi-currency rounding refactor in dep-7719 was deployed at 02:07, just minutes before the error spike at 02:11, and added JPY support, matching the failure symptom.
  - [

Ground truth, for comparison.

In [4]:
print(GT["inc-002"]["root_cause"]["summary"])

Release 2.14.0 (dep-7719) refactored money rounding and assumed every currency has two decimal places. JPY is a zero-decimal currency, so MoneyRounding.toMinorUnits throws ArithmeticException on every JPY authorisation. JPY is 5.4% of eu-west-1 traffic, which is why total 5xx sits at ~6% while EUR and GBP are untouched.


That is the shape of a good answer: it names the component, the mechanism, and
the change that introduced it.

## 2 · An incident built to punish it

`inc-008` is a worker being OOM-killed at the morning peak. Every obvious signal
says *memory pressure, raise the limit*. The real cause is nine days old and sits
outside the alert window.

In [5]:
rid_008, d008 = run("r01_diagnose", incident_id="inc-008")
show(d008, "r01 on inc-008")

r01 on inc-008
root cause  : A listener leak in the metrics hook introduced in dep-2201, which registers a new EventEmitter listener per render call without cleaning them up, causes unbounded memory growth in the notification-worker process until it exceeds the 1.5 Gi memory limit and gets OOMKilled.
remediation : Remove the listener registration from each render call in the metrics hook (introduced in dep-2201) and instead use a single persistent listener or a registry that is cleaned up after each render cycle. Alternatively, revert dep-2201 until a fixed version is deployed.
confident   : True
  - [critical] deploys.json: Deploy dep-2201 added a metrics hook that registers a listener per render call on the shared emitter, creating a memory leak.
  - [critical] metrics.json: Heap snapshot shows 1,841,203 EventEmitter[render] objects retaining 812 MB, confirming the listener leak.
  - [high    ] metrics.json: Daily max RSS per pod grew from 402 MB on July 1 to 1394 MB on July 14, and 

Now the ground truth, and the list of answers it marks as wrong.

In [6]:
g = GT["inc-008"]
print("actual root cause:\n ", g["root_cause"]["summary"], "\n")
print("correct remediation:\n ", g["correct_remediation"]["immediate"], "\n")
print("remediations marked WRONG:")
for w in g["wrong_remediations"]:
    print("  x", w)

actual root cause:
  Release 5.2.0 (dep-2201, nine days earlier) added a metrics hook that attaches a listener to a module-level EventEmitter on every render and never removes it. Each listener retains its RecipientContext, so RSS grows monotonically with cumulative messages processed rather than with concurrency. Daily peak RSS climbed from ~405MB to 1394MB over nine days and crossed the 1.5Gi limit during this morning's ordinary peak, producing the OOM kill loop. 

correct remediation:
  Roll back to 5.1.x, or ship a one-line fix that removes the listener after render (or uses a single module-level listener). A scheduled restart is an acceptable stopgap but must be labelled as one. 

remediations marked WRONG:
  x Raise the memory limit to 3Gi
  x Add more replicas
  x Roll back email-gateway
  x Add a nightly restart and close the incident


Score the run against ground truth rather than trusting how sure it sounds.

In [7]:
must = g["root_cause_must_include"]
hit = [k for k in must if k.lower() in d008["root_cause"].lower()]
print("root-cause keywords matched:", hit or "none", f"   (ground truth needs all of {must})")
bad = [w for w in g["wrong_remediations"] if w.split()[0].lower() in d008["remediation"].lower()]
print("remediation resembles a known-wrong one:", bad or "no")

root-cause keywords matched: ['leak', 'listener']    (ground truth needs all of ['leak', 'listener', '5.2.0'])
remediation resembles a known-wrong one: no


One shot is a coin flip on an incident like this. Sometimes it reaches nine days
back; sometimes it stops at the peak and says raise the limit. The two answers look
identical — same schema, same confidence — and nothing in the output tells you which
one you got.

Both runs, as graphs. One call each: a single node, depth 1.

In [8]:
dag.render_two(rid_002, rid_008, labels=("r01 · inc-002 (solved)", "r01 · inc-008 (missed)"))

```mermaid
flowchart LR
  subgraph ag["r01 · inc-002 (solved) — 1 exec · depth 1 · fan-out 0"]
  direction TD
    a0["r01_diagnose<br/><small>✓ succeeded · 13.5s</small>"]
    class a0 ok;
  end
  subgraph bg["r01 · inc-008 (missed) — 1 exec · depth 1 · fan-out 0"]
  direction TD
    b0["r01_diagnose<br/><small>✓ succeeded · 13.5s</small>"]
    class b0 ok;
  end
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

## What you learned

- A reasoner is dispatched over HTTP and returns a validated `Diagnosis`.
- One shot is enough when the evidence sits inside the alert window.
- One shot fails silently when the cause is outside it — confident, structured, wrong.

**Next:** `02_loop` — give the answer a critic, and let it try again.